# nb_03_gold_model
**Milestone 3: Gold layer.** Builds an analytics-ready **star schema** from Silver, for Power BI.

| Table | Grain (one row per) |
|---|---|
| `fact_sales` | order line |
| `dim_date` | calendar day |
| `dim_customer` | customer |
| `dim_product` | product |
| `dim_store` | store (incl. Online) |

### Cell 1: Parameters
Tolerance used when reconciling Gold revenue back to Silver.

In [1]:
revenue_tolerance_usd = 0.01   # max allowed difference when reconciling Gold revenue to Silver

StatementMeta(, dd1dcf24-4b8c-4e05-8650-d3d810413593, 3, Finished, Available, Finished, False)

### Cell 2: Setup
`data_end_date` = the last day with sales (20 Feb 2021). Ages, store ages and the partial-year flag are calculated relative to this date, so results don't change depending on which day you run the notebook.

In [2]:
import uuid
from pyspark.sql import functions as F

batch_id = str(uuid.uuid4())
s_sales     = spark.table("silver_sales")
s_customers = spark.table("silver_customers")
s_products  = spark.table("silver_products")
s_stores    = spark.table("silver_stores")
s_fx        = spark.table("silver_exchange_rates")

# Reference date = last day with sales. Used for ages and partial-year logic,
# so results don't change depending on the day the notebook runs.
data_end_date = s_sales.agg(F.max("order_date")).first()[0]
print("Data end date:", data_end_date)

def write_gold(df, table_name):
    (df.withColumn("_batch_id", F.lit(batch_id))
       .withColumn("_processed_at", F.current_timestamp())
       .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(table_name))
    print(f"{table_name:<16} {spark.table(table_name).count():>7,} rows")

StatementMeta(, dd1dcf24-4b8c-4e05-8650-d3d810413593, 4, Finished, Available, Finished, False)

Data end date: 2021-02-20


### Cell 3: dim_date
One row per day, covering full calendar years. `date_key` is an integer `yyyyMMdd`, the standard pattern for joining facts to dates.

- `year_month_num` lets Power BI sort `year_month` correctly.
- **`is_ytd_comparable`** is TRUE for days up to the same point in the year as the last sales day (Feb 20). Because 2021 only has 7 weeks of data, this flag is what makes a **fair year-over-year** comparison possible.

In [3]:
bounds = s_sales.agg(F.min("order_date").alias("lo"),
                     F.greatest(F.max("order_date"), F.max("delivery_date")).alias("hi")).first()
start, end = f"{bounds.lo.year}-01-01", f"{bounds.hi.year}-12-31"     # full calendar years

dim_date = (spark.sql(f"SELECT explode(sequence(DATE'{start}', DATE'{end}', INTERVAL 1 DAY)) AS date")
    .select(
        F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
        "date",
        F.year("date").alias("year"),
        F.quarter("date").alias("quarter_num"),
        F.concat(F.lit("Q"), F.quarter("date")).alias("quarter"),
        F.month("date").alias("month_num"),
        F.date_format("date", "MMMM").alias("month_name"),
        F.date_format("date", "MMM").alias("month_short"),
        F.date_format("date", "yyyy-MM").alias("year_month"),
        (F.year("date") * 100 + F.month("date")).alias("year_month_num"),   # sort key for year_month
        F.dayofmonth("date").alias("day_of_month"),
        ((F.dayofweek("date") + 5) % 7 + 1).alias("day_of_week_num"),      # Monday = 1 ... Sunday = 7
        F.date_format("date", "EEEE").alias("day_name"),
        F.weekofyear("date").alias("iso_week"),
        F.dayofweek("date").isin(1, 7).alias("is_weekend"),
        (F.col("date") <= F.lit(data_end_date)).alias("is_in_data_range"),
        # like-for-like flag: same point in the year as the last sales day (for fair YoY with partial 2021)
        (F.dayofyear("date") <= F.dayofyear(F.lit(data_end_date))).alias("is_ytd_comparable"),
    ))
write_gold(dim_date, "dim_date")

StatementMeta(, dd1dcf24-4b8c-4e05-8650-d3d810413593, 5, Finished, Available, Finished, False)

dim_date           2,192 rows


### Cell 4: dim_customer
Adds `age` (as of the data end date) and `age_band` for demographic analysis.

**Design note:** this uses the source `customer_key` directly as the dimension key, with no surrogate key, and overwrites changes (SCD Type 1). That's justified here: one source system, stable keys, and no requirement to track history.

In [4]:
age = F.floor(F.months_between(F.lit(data_end_date), F.col("birth_date")) / 12).cast("int")
dim_customer = (s_customers
    .withColumn("age", age)
    .withColumn("age_band", F.when(F.col("age") < 25, "18-24")
                             .when(F.col("age") < 35, "25-34")
                             .when(F.col("age") < 45, "35-44")
                             .when(F.col("age") < 55, "45-54")
                             .when(F.col("age") < 65, "55-64")
                             .otherwise("65+"))
    .select("customer_key", "customer_name", "gender", "age", "age_band", "birth_date",
            "city", "state_code", "state", "zip_code", "country", "continent"))
write_gold(dim_customer, "dim_customer")

StatementMeta(, dd1dcf24-4b8c-4e05-8650-d3d810413593, 6, Finished, Available, Finished, False)

dim_customer      15,266 rows


### Cell 5: dim_product
The product hierarchy Category → Subcategory → Product, plus unit margin. These are **current list prices**; the source has no price history.

In [5]:
dim_product = (s_products
    .withColumn("unit_margin_usd", F.col("unit_price_usd") - F.col("unit_cost_usd"))
    .withColumn("unit_margin_pct", F.round(F.col("unit_margin_usd") / F.col("unit_price_usd"), 4))
    .select("product_key", "product_name", "brand", "color",
            "subcategory_key", "subcategory", "category_key", "category",
            "unit_cost_usd", "unit_price_usd", "unit_margin_usd", "unit_margin_pct"))
write_gold(dim_product, "dim_product")

StatementMeta(, dd1dcf24-4b8c-4e05-8650-d3d810413593, 7, Finished, Available, Finished, False)

dim_product        2,517 rows


### Cell 6: dim_store
Adds a readable `store_name` and `store_age_years`. Online is store 0 with `store_type = 'Online'`.

In [6]:
dim_store = (s_stores
    .withColumn("store_name", F.when(F.col("store_type") == "Online", F.lit("Online Store"))
                               .otherwise(F.concat_ws(" - ", "country", "state")))
    .withColumn("store_age_years",
                F.floor(F.months_between(F.lit(data_end_date), F.col("open_date")) / 12).cast("int"))
    .select("store_key", "store_name", "store_type", "country", "state",
            "square_meters", "open_date", "store_age_years"))
write_gold(dim_store, "dim_store")

StatementMeta(, dd1dcf24-4b8c-4e05-8650-d3d810413593, 8, Finished, Available, Finished, False)

dim_store             67 rows


### Cell 7: fact_sales
Grain: **one row per order line**.

**Currency logic (the key business rule):**
- Product prices are already in **USD**, so `sales_amount_usd = quantity × unit_price_usd`.
- `exchange_rate` = **local currency per 1 USD** on the **order date**, so `sales_amount_local = sales_amount_usd × exchange_rate`.
- Converted **once** here, so every report uses identical numbers.

**Documented assumption:** amounts use current list prices, because the source has no historical prices or discounts.

In [7]:
fx = s_fx.select(F.col("rate_date").alias("order_date"), "currency_code", "exchange_rate")
price = s_products.select("product_key", "unit_price_usd", "unit_cost_usd")

fact_sales = (s_sales
    .join(price, "product_key", "left")
    .join(fx, ["order_date", "currency_code"], "left")
    .select(
        "order_number", "line_item",
        F.date_format("order_date", "yyyyMMdd").cast("int").alias("order_date_key"),
        F.date_format("delivery_date", "yyyyMMdd").cast("int").alias("delivery_date_key"),  # NULL in-store
        "customer_key", "store_key", "product_key",
        "currency_code", "is_online", "delivery_days", "quantity",
        "unit_price_usd", "unit_cost_usd", "exchange_rate",
        # USD amounts: prices are already in USD
        (F.col("quantity") * F.col("unit_price_usd")).cast("decimal(18,2)").alias("sales_amount_usd"),
        (F.col("quantity") * F.col("unit_cost_usd")).cast("decimal(18,2)").alias("cost_amount_usd"),
    )
    .withColumn("gross_profit_usd", (F.col("sales_amount_usd") - F.col("cost_amount_usd")).cast("decimal(18,2)"))
    # Local currency = USD x rate on the order date (rate = local currency per 1 USD)
    .withColumn("sales_amount_local", F.round(F.col("sales_amount_usd") * F.col("exchange_rate"), 2).cast("decimal(18,2)"))
    .withColumn("cost_amount_local",  F.round(F.col("cost_amount_usd")  * F.col("exchange_rate"), 2).cast("decimal(18,2)"))
    .withColumn("gross_profit_local", (F.col("sales_amount_local") - F.col("cost_amount_local")).cast("decimal(18,2)")))
write_gold(fact_sales, "fact_sales")

StatementMeta(, dd1dcf24-4b8c-4e05-8650-d3d810413593, 9, Finished, Available, Finished, False)

fact_sales        62,884 rows


### Cell 8: Gold quality gate
15 checks: grain uniqueness, orphan keys, dimension key uniqueness, and an **independent revenue reconciliation** that recomputes revenue from Silver with SQL and compares it to Gold. Any failure stops the run, so a pipeline won't refresh Power BI with bad numbers.

In [8]:
checks = []
def check(name, passed, detail):
    checks.append((name, "PASS" if passed else "FAIL", str(detail)))

f = spark.table("fact_sales")
n_fact, n_silver = f.count(), s_sales.count()
check("fact row count = silver_sales", n_fact == n_silver, f"{n_fact:,} vs {n_silver:,}")

dupes = f.groupBy("order_number", "line_item").count().filter("count > 1").count()
check("fact grain unique (order_number, line_item)", dupes == 0, f"{dupes} duplicates")

for col, dim, key in [("customer_key", "dim_customer", "customer_key"),
                      ("product_key",  "dim_product",  "product_key"),
                      ("store_key",    "dim_store",    "store_key"),
                      ("order_date_key", "dim_date",   "date_key")]:
    orphans = f.join(spark.table(dim).select(F.col(key).alias(col)), col, "left_anti").count()
    check(f"every {col} exists in {dim}", orphans == 0, f"{orphans} orphans")

late = (f.filter("delivery_date_key IS NOT NULL")
         .join(spark.table("dim_date").select(F.col("date_key").alias("delivery_date_key")), "delivery_date_key", "left_anti").count())
check("every delivery_date_key exists in dim_date", late == 0, f"{late} orphans")

for dim, key in [("dim_customer", "customer_key"), ("dim_product", "product_key"),
                 ("dim_store", "store_key"), ("dim_date", "date_key")]:
    d = spark.table(dim)
    check(f"{dim}.{key} unique", d.count() == d.select(key).distinct().count(), f"{d.count():,} rows")

nulls = f.filter("exchange_rate IS NULL OR unit_price_usd IS NULL").count()
check("no missing price or exchange rate", nulls == 0, f"{nulls} rows")

# Independent revenue reconciliation: recompute from Silver with a different method (SQL)
silver_rev = spark.sql("""
    SELECT SUM(s.quantity * p.unit_price_usd) AS rev
    FROM silver_sales s JOIN silver_products p ON s.product_key = p.product_key""").first().rev
gold_rev = f.agg(F.sum("sales_amount_usd")).first()[0]
check("revenue USD: Gold = Silver", abs(gold_rev - silver_rev) <= revenue_tolerance_usd,
      f"gold {gold_rev:,.2f} vs silver {silver_rev:,.2f}")

neg = f.filter("gross_profit_usd < 0").count()
check("no negative gross profit lines", neg == 0, f"{neg} lines")

usd_ok = f.filter("currency_code = 'USD' AND sales_amount_local <> sales_amount_usd").count()
check("USD orders: local amount = USD amount", usd_ok == 0, f"{usd_ok} mismatches")

checks_df = spark.createDataFrame(checks, "check string, status string, detail string")
display(checks_df)
failed = [c[0] for c in checks if c[1] == "FAIL"]
if failed:
    raise Exception(f"Gold quality gate FAILED: {failed}")
print(f"Gold complete. Batch {batch_id}")

StatementMeta(, dd1dcf24-4b8c-4e05-8650-d3d810413593, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, af1aa44b-0738-4e7e-a454-9e00ed81e799)

Gold complete. Batch 40c8fa87-de6a-4045-a452-cebc81eb4eee


---
## Validation and first business insights
### V1: Revenue, profit and margin by year
Expected: total ≈ **$55.76M**; 2019 peak ≈ $18.26M; 2020 ≈ $9.29M; margin steady around **58–59%**.

In [9]:
%%sql
SELECT d.year, COUNT(DISTINCT f.order_number) AS orders, ROUND(SUM(f.sales_amount_usd),0) AS revenue_usd,
       ROUND(SUM(f.gross_profit_usd),0) AS profit_usd, ROUND(SUM(f.gross_profit_usd)/SUM(f.sales_amount_usd)*100,1) AS margin_pct
FROM fact_sales f JOIN dim_date d ON f.order_date_key = d.date_key GROUP BY d.year ORDER BY d.year

StatementMeta(, dd1dcf24-4b8c-4e05-8650-d3d810413593, 11, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 5 fields>

### V2: Like-for-like comparison (1 Jan – 20 Feb each year)
The full-year chart makes 2021 look like a 89% collapse, but that's only because 2021 has 7 weeks of data. Comparing the **same period**: 2021 ≈ $1.04M vs 2020 ≈ $3.49M, a real but much smaller **–70%** drop. Note also that early 2020 was *higher* than early 2019, so the 2020 decline happened **later in the year**.

In [10]:
%%sql
SELECT d.year, ROUND(SUM(f.sales_amount_usd),0) AS revenue_jan1_to_feb20
FROM fact_sales f JOIN dim_date d ON f.order_date_key = d.date_key WHERE d.is_ytd_comparable GROUP BY d.year ORDER BY d.year

StatementMeta(, dd1dcf24-4b8c-4e05-8650-d3d810413593, 12, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 2 fields>

### V3: Revenue and margin by category
Expected: Computers is #1 (≈ $19.3M), Home Appliances #2. Games and Toys has the lowest margin (≈ 54.7%).

In [11]:
%%sql
SELECT p.category, ROUND(SUM(f.sales_amount_usd),0) AS revenue_usd, ROUND(SUM(f.gross_profit_usd)/SUM(f.sales_amount_usd)*100,1) AS margin_pct
FROM fact_sales f JOIN dim_product p ON f.product_key = p.product_key GROUP BY p.category ORDER BY revenue_usd DESC

StatementMeta(, dd1dcf24-4b8c-4e05-8650-d3d810413593, 13, Finished, Available, Finished, False)

<Spark SQL result set with 8 rows and 3 fields>

### V4: The currency conversion
Expected: USD local = USD amount. EUR and GBP local amounts are *smaller* than USD (stronger currencies); CAD and AUD are *larger*.

In [12]:
%%sql
SELECT f.currency_code, ROUND(SUM(f.sales_amount_usd),0) AS revenue_usd, ROUND(SUM(f.sales_amount_local),0) AS revenue_local
FROM fact_sales f GROUP BY f.currency_code ORDER BY revenue_usd DESC

StatementMeta(, dd1dcf24-4b8c-4e05-8650-d3d810413593, 14, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 3 fields>

### V5: Hand-check one order
Expected: order 366000, CAD, 1 × $68.00, rate 1.3884, giving **CAD 94.41**. Check it with a calculator: 68 × 1.3884 = 94.41.

In [13]:
%%sql
SELECT f.order_number, f.line_item, f.currency_code, f.quantity, f.unit_price_usd, f.exchange_rate, f.sales_amount_usd, f.sales_amount_local
FROM fact_sales f WHERE f.order_number = 366000

StatementMeta(, dd1dcf24-4b8c-4e05-8650-d3d810413593, 15, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 8 fields>